In [ ]:
"""
Preprocessing for the NPD_eORCA025 wind-perturbation attribution experiments, their
control equivalents, the full-length climatological baselines (NPD control 1996-2023; 
HadGEM3-GC31-HH control-1950 1976-2005), and the single-year HG3-1995 from the coupled 
model simulation HadGEM3-GC31-HH control-1950 of CMIP6 HighResMIP.

Note: NPD stands for Near-Present-Day, the model configuration used is part of this NPD 
model suite (refer to the manuscript/README for more details). HG3 stands for HadGEM3-GC31-HH.

Processing in this script includes:
-------------------------------------
1. Building climatological baseline averages of conservative temperature, per calendar month,
over the full run period: BASELINE_NPD (monthly + daily, 1996-2023) and BASELINE_HG3 (monthly,
1976-2005). Leap days are dropped from the daily files first before averaging.

2. Geographically subsetting conservative temperature (monthly + daily) over the Tropical
Pacific and the standard Niño boxes (1+2, 3, 3.4, 4) via CDO, for every experiment and control 
equivalents, climatological baseline averaged files and HG3-1995. These files are then saved as
individual NetCDF files per month.

3. Calculating the depth of the 20˚C isotherm (Z20), merged into one file per run; the per-month 
Niño-box crops are likewise merged into whole-year files.

4. Computing daily volume and heat budgets over depth ranges (0-60 m, 60-120 m, 120-180 m) in
the central-eastern Pacific, for the wind-perturbation experiments, their controls and the
1996-2023 NPD control baseline (not for BASELINE_HG3 or HG3-1995, which have no daily output).

-------------------------------------------------------------------------------------------
Author: Sreevathsa G. (sg13n23@soton.ac.uk; ORCID ID: 0000-0003-4084-9677)
Last updated: 21 September 2026
"""

In [ ]:
# Import statements
import glob
import os

import numpy as np
import xarray as xr
from tqdm.notebook import tqdm

from nemo_box_budget import compute_box_budgets
from utils import renamer, setup_dask_cluster, z20_calculator

from cdo import Cdo
cdo = Cdo(tempdir='/dssgfs01/scratch/sg13n23/ATTRIBUTION_EXPS/NPD_Wind_Exp_Diagnostics/tmp/')

# Setup Dask cluster for parallel computation of budgets.
cluster, client = setup_dask_cluster()
print('Dashboard URL:', client.dashboard_link)

In [ ]:
# Setup paths and directories
PARENT_DIR = '/dssgfs01/scratch/sg13n23/ATTRIBUTION_EXPS/NOC_Near_Present_Day/nemo/cfgs/GLOBAL_QCO/'
BASE_DIR   = '/dssgfs01/scratch/sg13n23/ATTRIBUTION_EXPS/NPD_Wind_Exp_Diagnostics/'

# PARENT_PATH is where all the single-year model runs and long-term runs are stored.

# Dictionary of experiments, control equivalents and full-length climatological baseline runs 
# along with their corresponding experiment/internal IDs (i.e., labels used in the file names).
EXPS = {'ANWSUP'        : 'EXP2023_35',
        'ANWTRN-JRA'    : 'EXP2013_01',
        'ANWTRN-HG3'    : 'EXP2013_03',
        'WNDREP-HG3'    : 'EXP2013_04_c1',
        'WNDREP-HG3c'   : 'EXP2013_06_c1',
        'CTRL_2013'     : 'EXP2013_CTRL',
        'CTRL_2023'     : 'EXP2023_CTRL',
        'BASELINE_HG3'  : 'BASELINE_HG3',
        'BASELINE_NPD'  : 'BASELINE_NPD',
        'HG3-1995'      : 'HG3-1995',}

# Month range for the analysis (1-12 for Jan-Dec)
months = np.arange(1, 13, 1)

In [ ]:
# =============================================================================
# CALCULATING CLIMATOLOGICAL BASELINE AVERAGES FOR NPD MODEL RUN (BASELINE_NPD: 
# 1996-2023) AND HADGEM3-GC31-HH CONTROL-1950 SIMULATION (BASELINE_HG3: 1976-2005)
# =============================================================================
'''
Note: For conservative temperature, the variable name 'toce_con' and 'thetao_con' are both used in the datasets. 
This was mainly due to how the file-defs were set up in the NEMO model configuration. In the end when plotting and analysis, 
the 'renamer' utility function is used to standardize the variable/coordinate names so that there aren't any conflicts when 
conducting operations on two datasets from different sources. 
'''

# BASELINE_HG3: 1m (monthly-mean files) only, average 1976-2005
# ------------------------------------------------------------------------------
'''
Preprocessing note: For the HadGEM3-GC31-HH control-1950 simulation (HG3), conservative temperature was calculated from potential 
temperature and salinity (provided as variables in T-grid model output) using the TEOS-10 Gibbs Seawater (GSW) Oceanographic 
Python Toolbox (https://teos-10.github.io/GSW-Python/). These conversion steps were performed in a different HPC than where this 
repository was developed; please reach out to the author if you have any questions about these preprocessing steps.
'''
# Create directory for storing the climatological baseline averages for HadGEM3-GC31-HH control-1950 simulation (HG3)
os.makedirs(BASE_DIR + 'HadGEM3-GC31-HH/BASELINE/', exist_ok=True)
for m in months: # Loop through each month (1-12)
    # List of file names of HG3 for the given month (m) and years 1976-2005
    fnames = [BASE_DIR + f'HadGEM3-GC31-HH/{year}/HadGEM3-GC31-HH_thetao_con_control-1950_1m_y{year}m{m:02d}.nc' for year in range(1976, 2006)]
    for f in fnames: # Loop through each file for the given month
        # Open the dataset and extract the 'toce_con' variable (conservative temperature), after renaming the coordinates/dimensions/variable names 
        # for standardised names using the 'renamer' utility function. Originally, 'toce_con' is 'thetao_con' in HG3 files, but is renamed by 'renamer'.
        ds = renamer(xr.open_dataset(f))['toce_con']
        if fnames.index(f) == 0: # If it's the first file, initialize the mean with this dataset
            mean = ds
        else: # For subsequent files, add the dataset to the mean after aligning the time coordinate
            ds['time'] = mean['time']
            mean = mean + ds
    # After processing all files for the month, divide by the number of files to get the average
    mean = mean / len(fnames)
    # Save the climatological mean to a NetCDF file for the given month
    mean.to_netcdf(BASE_DIR + f'HadGEM3-GC31-HH/BASELINE/HadGEM3-GC31-HH_control-1950_BS1976-2005_1m_M{m:02d}.nc')

# BASELINE_NPD 1m (monthly-mean files): average 1996-2023
# ------------------------------------------------------------------------------
# Create directory for storing the climatological baseline averages for NPD control simulation
os.makedirs(BASE_DIR + 'CTRL_BASELINE/', exist_ok=True)
for m in months: # Loop through each month (1-12)
    # List of file names of the simulation for the given month (m) and years 1996-2023 (full 3D run period)
    fnames = sorted(glob.glob(PARENT_DIR + f'BASELINE_NPD/OUTPUT/eORCA025_1m_grid_T_????{m:02d}-????{m:02d}.nc'))
    for f in fnames:
        # Open the dataset and extract the 'thetao_con' variable (conservative temperature).
        ds = xr.open_dataset(f)['thetao_con']
        if fnames.index(f) == 0:
            mean = ds
        else:
            ds['time_counter'] = mean['time_counter']
            mean = mean + ds
    mean = mean / len(fnames)
    # Save the climatological mean to a NetCDF file for the given month
    mean.to_netcdf(BASE_DIR + f'CTRL_BASELINE/CTRL_RUN_eORCA025_JRA55_BS1996-2023_1m_M{m:02d}.nc')

# BASELINE_NPD 1d (daily-mean files): average 1996-2023, dropping leap day first
# ------------------------------------------------------------------------------
for m in months: # Loop through each month (1-12)
    # List of file names of the simulation for the given month (m) and years 1996-2023 (full 3D run period)
    fnames = sorted(glob.glob(PARENT_DIR + f'BASELINE_NPD/OUTPUT/eORCA025_1d_gridT_ENSO_????{m:02d}-????{m:02d}.nc'))
    for f in fnames:
        # Open the dataset and extract the 'toce_con' variable (conservative temperature).
        ds = xr.open_dataset(f)['toce_con']
        ds = ds[~((ds.time_counter.dt.month == 2) & (ds.time_counter.dt.day == 29))]
        if fnames.index(f) == 0:
            mean = ds
        else:
            ds['time_counter'] = mean['time_counter']
            mean = mean + ds
    mean = mean / len(fnames)
    # Save the climatological mean to a NetCDF file for the given month
    mean.to_netcdf(BASE_DIR + f'CTRL_BASELINE/CTRL_RUN_eORCA025_JRA55_BS1996-2023_1d_M{m:02d}.nc')

In [ ]:
# =============================================================================
# GEOGRAPHICAL SUBSETTING OF CONSERVATIVE TEMPERATURE FILES (1m and 1d)
# =============================================================================
# The files being processed here are the monthly (1m) and daily (1d) conservative 
# temperature data saved in monthly files for each of the experiments, their control 
# equivalents, the climatological baseline averages (BASELINE_NPD and BASELINE_HG3), 
# and the single-year HG3-1995. The subsetting is done over the Tropical Pacific 
# region and the Nino monitoring boxes (N12, N3, N34, N4). The subsetting is 
# performed using the CDO (Climate Data Operators) tool. Finally, the depth of 
# the 20˚C isotherm (Z20) is calculated for the 1m and 1d files and merged into a 
# single file per experiment/control run.
# ------------------------------------------------------------------------------

# Defining the geographical boundaries of the standard Niño boxes
nino_boxes = {'N12': '-90,-80,-10,0',
              'N3': '-150,-90,-5,5',
              'N34': '-170,-120,-5,5',
              'N4': '160,210,-5,5'}

# Making directories for storing the subsetted data, if they don't exist already.
for sub in ['TOCE_CON', 'NINO_BOXES', 'Z20']:
    os.makedirs('./data/' + sub, exist_ok=True)

for exp in tqdm(EXPS.keys(), desc='Experiments'): # Loop through each item in the EXPS dictionary

    exp_id = EXPS[exp] # Get the corresponding experiment/internal ID

    # Input-file path templates (one {m:02d} placeholder each). Regular experiments
    # live under PARENT_DIR with the year in their ID; the baselines/HG3-1995 have
    # their own directories and naming. tmpl_1d = None  ->  no daily (1d) data.
    # tvar_* is the temperature variable to crop; only the NEMO experiments carry e3t.
    if exp_id.startswith('EXP'):
        year = int(exp_id[3:7])
        tmpl_1m = f'{PARENT_DIR}{exp_id}/OUTPUT/eORCA025_1m_grid_T_{year}{{m:02d}}-{year}{{m:02d}}.nc'
        tmpl_1d = f'{PARENT_DIR}{exp_id}/OUTPUT/eORCA025_1d_gridT_ENSO_{year}{{m:02d}}-{year}{{m:02d}}.nc'
        tvar_1m, tvar_1d, e3t = 'thetao_con', 'toce_con', ',e3t'
    elif exp == 'BASELINE_NPD':
        tmpl_1m = BASE_DIR + 'CTRL_BASELINE/CTRL_RUN_eORCA025_JRA55_BS1996-2023_1m_M{m:02d}.nc'
        tmpl_1d = BASE_DIR + 'CTRL_BASELINE/CTRL_RUN_eORCA025_JRA55_BS1996-2023_1d_M{m:02d}.nc'
        tvar_1m, tvar_1d, e3t = 'thetao_con', 'toce_con', ''
    elif exp == 'BASELINE_HG3':
        tmpl_1m = BASE_DIR + 'HadGEM3-GC31-HH/BASELINE/HadGEM3-GC31-HH_control-1950_BS1976-2005_1m_M{m:02d}.nc'
        tmpl_1d = None
        tvar_1m, e3t = 'toce_con', ''
    elif exp == 'HG3-1995':
        tmpl_1m = BASE_DIR + 'HadGEM3-GC31-HH/1995/HadGEM3-GC31-HH_thetao_con_control-1950_1m_y1995m{m:02d}.nc'
        tmpl_1d = None
        tvar_1m, e3t = 'toce_con', ''

    # Flag to indicate whether daily (1d) data is available for the current exp_id
    has_1d = tmpl_1d is not None

    # z20 lists: one entry per month (1d or 1m), merged into a single dataset after the loop
    z20_1d, z20_1m = [], []

    # Loop through each month (1-12) and perform geographical subsetting and Z20 calculation
    for m in tqdm(months, desc=f'Months ({exp})', leave=False):
        # Geographical subsetting for 1m (monthly-mean) files
        # ---------------------------------------------------
        inp_file_1m = tmpl_1m.format(m=m)
        # Subsetting over the Tropical Pacific region (120E-290E, 30S-30N) and saving to a new NetCDF file
        cdo.sellonlatbox('120,290,-30,30', input=f'-selvar,{tvar_1m}{e3t} ' + inp_file_1m,
                          output='./data/TOCE_CON/TOCE_CON_{exp}_1m_M{m:02d}.nc'.format(exp=exp, m=m))
        for nb, box in nino_boxes.items(): # Loop through each of the standard Niño boxes
            cdo.sellonlatbox(box, input=f'-selvar,{tvar_1m} ' + inp_file_1m,
                              output='./data/NINO_BOXES/{nb}_TOCE_CON_{exp}_1m_M{m:02d}.nc'.format(nb=nb, exp=exp, m=m))
        # Calculate Z20 for the subsetted 1m file and append to the z20_1m list
        z20_1m += [z20_calculator(renamer(xr.open_dataset('./data/TOCE_CON/TOCE_CON_{exp}_1m_M{m:02d}.nc'.format(exp=exp, m=m))))]

        # Geographical subsetting for 1d (daily-mean) files, if available.
        # ---------------------------------------------------------------
        # 1d files skipped for runs with no daily data (BASELINE_HG3, HG3-1995)
        if has_1d:
            inp_file_1d = tmpl_1d.format(m=m)
            # Subsetting over the Tropical Pacific region (130E-270E, 20S-20N) and saving to a new NetCDF file
            cdo.sellonlatbox('130,270,-20,20', input=f'-selvar,{tvar_1d}{e3t} ' + inp_file_1d,
                              output='./data/TOCE_CON/TOCE_CON_{exp}_1d_M{m:02d}.nc'.format(exp=exp, m=m))
            for nb, box in nino_boxes.items(): # Loop through each of the standard Niño boxes
                cdo.sellonlatbox(box, input=f'-selvar,{tvar_1d} ' + inp_file_1d,
                                  output='./data/NINO_BOXES/{nb}_TOCE_CON_{exp}_1d_M{m:02d}.nc'.format(nb=nb, exp=exp, m=m))
            # Calculate Z20 for the subsetted 1d file and append to the z20_1d list
            z20_1d += [z20_calculator(renamer(xr.open_dataset('./data/TOCE_CON/TOCE_CON_{exp}_1d_M{m:02d}.nc'.format(exp=exp, m=m))))]
    # Merge the monthly Z20 datasets (1m and 1d) into a single dataset for the whole year and save to NetCDF
    z20_1m = xr.merge(z20_1m)
    z20_1m.to_netcdf('./data/Z20/Z20_{exp}_1m_M01-12.nc'.format(exp=exp))
    if has_1d:
        z20_1d = xr.merge(z20_1d)
        z20_1d.to_netcdf('./data/Z20/Z20_{exp}_1d_M01-12.nc'.format(exp=exp))

# Merge per-month NINO_BOXES crops into whole-year files. Run once, after the loops.
for exp in tqdm(EXPS): # Loop through each item in the EXPS dictionary
    for nb in tqdm(nino_boxes, leave = False): # Loop through each of the standard Niño boxes
        for freq in ('1m', '1d'):# Loop through each frequency (monthly and daily)
            # List of files for the given box, experiment, and frequency (1m or 1d)
            files = sorted(glob.glob(f'./data/NINO_BOXES/{nb}_TOCE_CON_{exp}_{freq}_M*.nc'))
            if files: # If there are files for the given box, experiment, and frequency, merge them into a single file
                cdo.mergetime(input=' '.join(files),
                              output=f'./data/NINO_BOXES/{nb}_TOCE_CON_{exp}_{freq}.nc')

In [ ]:
# ===============================================================================
# VOLUME AND HEAT BUDGET COMPUTATION ON DAILY (1d) DATA FOR THE WIND-PERTURBATION 
# EXPERIMENTS AND THE CLIMATOLOGICAL BASELINE FROM THE 30-YR CONTROL RUN
# ===============================================================================
'''
Note: These budget calculations are not performed on BASELINE_HG3 and HG3-1995
due to the lack of daily (1d) output files for these runs.
'''
# -------------------------------------------------------------------------------

# Create output directory for storing the volume and heat budget results, if they don't exist already.
vol_heat_budgets_outdir = f'./data/VOL_HEAT_BUDGETS/'
os.makedirs(vol_heat_budgets_outdir, exist_ok=True)
# Load the domain mask for the zoomed region of interest configured in the experiment setup (y: 603-766, x: 69-860), 
# this can be found in the ./data/EXP_SETUP_FILES/grid_def_zoom1_EXPS.xml file. These indices (referenced to a standard 
# NEMO eORCA025 grid) are found manually and corresponding to the Tropical Pacific geographic area (20S-20N, 90E-70W).
domain = xr.open_dataset('./data/mesh_mask.nc').isel(x=slice(69, 69+792), y=slice(603, 603+164)).squeeze()

# Budget computation for each single-year experiment
# ---------------------------------------------------

for exp in EXPS.keys(): # Loop through each item in the EXPS dictionary
    exp_id = EXPS[exp]
    if not exp_id.startswith('EXP'): # Skip BASELINE_NPD (separately calculated below); BASELINE_HG3, HG3-1995 - No 1d output files for these runs
        continue
    # Load the 1D output files for T, U, V, W for the current experiment. The file paths are constructed based on the experiment ID and the grid type (T, U, V, W).
    fpath = PARENT_DIR + exp_id+ '/OUTPUT/eORCA025_1d_grid{GRID}_ENSO_'+f'{exp_id[3:7]}??-{exp_id[3:7]}??.nc'
    dst_1d = xr.open_mfdataset(sorted(glob.glob(fpath.format(GRID = 'T'))))
    dsu_1d = xr.open_mfdataset(sorted(glob.glob(fpath.format(GRID = 'U'))))
    dsv_1d = xr.open_mfdataset(sorted(glob.glob(fpath.format(GRID = 'V'))))
    dsw_1d = xr.open_mfdataset(sorted(glob.glob(fpath.format(GRID = 'W'))))
    # Compute budgets for different depth ranges (0-60m, 60-120m, 120-180m) for the current experiment. The compute_box_budgets function (from nemo_box_budget.py) 
    # is called with the appropriate parameters, including the domain mask and depth bounds.
    budgets_0_60m = compute_box_budgets(dst_1d, dsu_1d, dsv_1d, dsw_1d, domain, lat_bounds=(-5, 5), lon_bounds=(-170, -120), depth_bounds=(0, 62), interp_T = True).expand_dims(exp = [exp])
    budgets_60_120m = compute_box_budgets(dst_1d, dsu_1d, dsv_1d, dsw_1d, domain, lat_bounds=(-5, 5), lon_bounds=(-170, -120), depth_bounds=(60, 120), interp_T = True).expand_dims(exp = [exp])
    budgets_120_180m = compute_box_budgets(dst_1d, dsu_1d, dsv_1d, dsw_1d, domain, lat_bounds=(-5, 5), lon_bounds=(-170, -120), depth_bounds=(120, 181), interp_T = True).expand_dims(exp = [exp])
    # Concatenate the budgets for different depth ranges into a single xarray dataset and save to NetCDF
    budgets = xr.concat([budgets_0_60m.expand_dims(depth_range = ['0-60']), budgets_60_120m.expand_dims(depth_range = ['60-120']), budgets_120_180m.expand_dims(depth_range = ['120-180'])], dim = 'depth_range')
    # Save the budgets to a NetCDF file in the output directory, with a filename that includes the experiment name and the year range.
    budgets.to_netcdf(f'{vol_heat_budgets_outdir}/BUDGETS_{exp}_{exp_id[3:7]}0101-{exp_id[3:7]}1231.nc')


# Budget computation for the climatological baseline from the 30-yr NPD control run from 1996-2023 (BASELINE_NPD)
# ---------------------------------------------------------------------------------------------------------------

# Load the domain mask for the zoomed region of interest (y: 559-811, x: 227-860), this can be found in the ./data/EXP_SETUP_FILES/grid_def_zoom1_BASELINE.xml file.
# These indices (referenced to a standard NEMO eORCA025 grid) are found manually and corresponding to the Tropical Pacific geographic area (30S-30N, 130E-70W).
# There is a slight discrepancy (not meaninfully affecting the analysis) in the y,x indices between the zoom output of the experiments and the long-term control run from 1996-2023. 
# Therefore, domain is redeclared and subsetted over the appropriate x,y indices for the long-term control run. 
domain = xr.open_dataset('./data/mesh_mask.nc').isel(x=slice(227, 227+634), y=slice(559, 559+253)).squeeze()

# Load the 1D output files for T, U, V, W for the full 1996-2023 run period of the control run. The file paths are constructed based on the grid type (T, U, V, W), with the 
# monthly files covering the whole run period opened together as a single continuous daily time series.
fpath  = PARENT_DIR + 'BASELINE_NPD/OUTPUT/eORCA025_1d_grid{GRID}_ENSO_??????-??????.nc'
dst_1d = xr.open_mfdataset(sorted(glob.glob(fpath.format(GRID = 'T'))))
dsu_1d = xr.open_mfdataset(sorted(glob.glob(fpath.format(GRID = 'U'))))
dsv_1d = xr.open_mfdataset(sorted(glob.glob(fpath.format(GRID = 'V'))))
dsw_1d = xr.open_mfdataset(sorted(glob.glob(fpath.format(GRID = 'W'))))
# Compute budgets for different depth ranges (0-60m, 60-120m, 120-180m) for the climatological baseline from the 30-yr control run. The compute_box_budgets function 
# is called with the appropriate parameters, including the domain mask and depth bounds.
budgets_0_60m = compute_box_budgets(dst_1d, dsu_1d, dsv_1d, dsw_1d, domain, lat_bounds=(-5, 5), lon_bounds=(-170, -120), depth_bounds=(0, 62), interp_T = True).expand_dims(exp = ['BASELINE_NPD'])
budgets_60_120m = compute_box_budgets(dst_1d, dsu_1d, dsv_1d, dsw_1d, domain, lat_bounds=(-5, 5), lon_bounds=(-170, -120), depth_bounds=(60, 120), interp_T = True).expand_dims(exp = ['BASELINE_NPD'])
budgets_120_180m = compute_box_budgets(dst_1d, dsu_1d, dsv_1d, dsw_1d, domain, lat_bounds=(-5, 5), lon_bounds=(-170, -120), depth_bounds=(120, 181), interp_T = True).expand_dims(exp = ['BASELINE_NPD'])
# Concatenate the budgets for different depth ranges into a single xarray dataset and save to NetCDF
budgets = xr.concat([budgets_0_60m.expand_dims(depth_range = ['0-60']), budgets_60_120m.expand_dims(depth_range = ['60-120']), budgets_120_180m.expand_dims(depth_range = ['120-180'])], dim = 'depth_range')
# Save the budgets to a NetCDF file in the output directory, with a filename that includes the baseline name and the year range.
budgets.to_netcdf(f'{vol_heat_budgets_outdir}/BUDGETS_BASELINE_NPD_1996-2023.nc')